In [ ]:
import subprocess

subprocess.run([
    "pip", "install", "--force-reinstall",
    "transformers==4.44.0",
    "tokenizers==0.19.1",
    "huggingface-hub==0.24.0",
    "peft==0.11.1",
    "accelerate==0.30.1",
], check=True)

print("✅ Done! Runtime → Restart Session NOW")

## RESTART SESSION before Cell 2

In [ ]:
!pip install -q pymupdf
!pip install -q langchain-text-splitters
!pip install -q datasets
!pip install -q rank_bm25
!pip install -q requests

import subprocess
subprocess.run([
    "pip", "install", "--no-deps",
    "git+https://github.com/FlagOpen/FlagEmbedding.git"
], check=True)

import transformers, huggingface_hub, torch
from FlagEmbedding import FlagModel

print(f"✅ transformers   : {transformers.__version__}")
print(f"✅ huggingface-hub: {huggingface_hub.__version__}")
print(f"✅ GPU            : {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM           : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"✅ FlagModel      : OK")

assert transformers.__version__ == "4.44.0", f"❌ Wrong: {transformers.__version__}"
print("\n✅ All versions correct! Continue.")

In [ ]:
import subprocess, time, requests, os

print("🔄 Downloading Ollama...")
subprocess.run([
    "curl", "-L",
    "https://github.com/ollama/ollama/releases/download/v0.3.12/ollama-linux-amd64.tgz",
    "-o", "/tmp/ollama.tgz"
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

subprocess.run(["tar", "-xzf", "/tmp/ollama.tgz", "-C", "/tmp/"],
               check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

result = subprocess.run(
    ["find", "/tmp", "-name", "ollama", "-type", "f"],
    capture_output=True, text=True
)
binary = result.stdout.strip().split('\n')[0]
subprocess.run(["cp", binary, "/usr/local/bin/ollama"], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/ollama"],  check=True)

ver = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
print(f"✅ {ver.stdout.strip()}")

os.environ["OLLAMA_HOST"]    = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["HOME"]           = "/root"

print("🔄 Starting Ollama server...")
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=os.environ.copy()
)

for i in range(15):
    try:
        requests.get("http://localhost:11434", timeout=3)
        print("✅ Ollama server running!")
        break
    except:
        print(f"   Waiting... ({i+1}/15)")
        time.sleep(3)

In [ ]:
import subprocess, requests, time

OLLAMA_MODEL = "llama3.1:8b"

print(f"🔄 Pulling {OLLAMA_MODEL} — takes 3-5 mins...")
subprocess.run(["ollama", "pull", OLLAMA_MODEL], capture_output=True, text=True)
print("✅ Model ready!")

models = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(f"Available:\n{models.stdout}")

def ask_ollama(prompt, max_tokens=1000, temperature=0.3):
    try:
        resp = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model"  : OLLAMA_MODEL,
                "prompt" : prompt,
                "stream" : False,
                "options": {
                    "num_predict"   : max_tokens,
                    "temperature"   : temperature,
                    "top_p"         : 0.9,
                    "repeat_penalty": 1.1,
                    "stop"          : ["Human:", "User:", "\n\n\n"]
                }
            },
            timeout=180
        )
        return resp.json()["response"].strip()
    except Exception as e:
        return None

resp = ask_ollama("What does GPON stand for? One sentence.", max_tokens=80)
print(f"\n🧪 Test: {resp}")
print("✅ Ollama ready!")

In [ ]:
import os

HF_TOKEN      = "hf_XXXXXXXXXXXXXXXX"
HF_USERNAME   = "your-username"
HF_MODEL_NAME = "bge-large-en-v1.5-nokia-finetuned"

# ⚠️ Upload your PDF to Colab before running
PDF_PATH = "/content/Alarms_Guide_3HH13538AAAATCZZA19_1.pdf"

CHUNK_SIZE         = 700
CHUNK_OVERLAP      = 150
MIN_CHUNK_LEN      = 150

QUERIES_PER_CHUNK  = 3
HARD_NEG_PER_QUERY = 7
BM25_NEGS          = 3
DENSE_NEGS         = 4
MAX_RETRIES        = 3

TRAIN_EPOCHS       = 10
BATCH_SIZE         = 4
GRAD_ACCUM         = 8
LEARNING_RATE      = 8e-6
MAX_QUERY_LEN      = 96
MAX_PASSAGE_LEN    = 512
TEMPERATURE        = 0.02
WARMUP_RATIO       = 0.1

OUTPUT_DIR         = "/content/finetuned-bge-nokia"
QUERY_INSTRUCTION  = "Represent this sentence for searching relevant passages:"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Config ready!")
print(f"   Effective batch : {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Epochs          : {TRAIN_EPOCHS}")
print(f"   Ollama model    : {OLLAMA_MODEL}")

In [ ]:
import fitz, re, json
from langchain_text_splitters import RecursiveCharacterTextSplitter

def extract_pdf(path):
    doc, pages = fitz.open(path), []
    for page in doc:
        text = page.get_text("text").strip()
        if len(text) > 80:
            pages.append(text)
    doc.close()
    return pages

def clean_text(text):
    text = re.sub(r'\f',             '\n',   text)
    text = re.sub(r'\n{3,}',         '\n\n', text)
    text = re.sub(r'-\s*\n\s*',      '',     text)
    text = re.sub(r'\n(?=[a-z,;])',  ' ',    text)
    text = re.sub(r'[ \t]{2,}',      ' ',    text)
    return text.strip()

print(f"📄 Reading: {PDF_PATH}")
pages     = extract_pdf(PDF_PATH)
full_text = clean_text("\n\n".join(pages))
print(f"   Pages      : {len(pages)}")
print(f"   Characters : {len(full_text):,}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    separators    = ["\n\n", "\n", ". ", "! ", "? ", ", ", " "]
)

chunks = splitter.split_text(full_text)
chunks = [c.strip() for c in chunks if len(c.strip()) >= MIN_CHUNK_LEN]
chunks = list(dict.fromkeys(chunks))

print(f"✅ Unique chunks: {len(chunks)}")
print(f"\n── Sample chunk ──\n{chunks[10][:400]}\n")

# Backup immediately
with open("/content/chunks_backup.jsonl", "w") as f:
    for i, c in enumerate(chunks):
        f.write(json.dumps({"idx": i, "text": c}) + "\n")
print("✅ Chunks backed up!")

In [ ]:
import json, time, re, random, os

PAIRS_FILE = "/content/pairs_progress.jsonl"

pairs        = []
done_indices = set()

if os.path.exists(PAIRS_FILE):
    with open(PAIRS_FILE) as f:
        for line in f:
            try:
                d = json.loads(line.strip())
                pairs.append((d["query"], d["chunk"]))
                done_indices.add(d["chunk_idx"])
            except:
                pass
    print(f"📂 Resuming — {len(pairs)} pairs, {len(done_indices)} chunks done")
else:
    print("🆕 Starting fresh")

def build_prompt(chunk, n):
    return f"""You are a Nokia ISAM telecom expert.

Read the passage and generate exactly {n} technical Q&A pairs.

RULES:
- Questions must be answerable COMPLETELY from the passage
- Cover different aspects: alarm causes, severity, hardware, procedures
- "pos" = exact relevant sentence(s) from the passage
- Return ONLY a JSON array — no markdown, no explanation

Format: [{{"query": "...", "pos": "..."}}]

Passage:
{chunk}

JSON:"""

def extract_json(text):
    if not text:
        return []
    try:
        r = json.loads(text)
        if isinstance(r, list):
            return r
    except:
        pass
    match = re.search(r'\[[\s\S]*?\]', text)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass
    objects = re.findall(r'\{[^{}]+\}', text)
    result  = []
    for obj in objects:
        try:
            result.append(json.loads(obj))
        except:
            pass
    return result

def generate_pairs_for_chunk(chunk, n=3, retries=MAX_RETRIES):
    for attempt in range(retries):
        raw    = ask_ollama(build_prompt(chunk, n), max_tokens=900, temperature=0.3)
        parsed = extract_json(raw)
        valid  = [
            (p["query"], chunk)
            for p in parsed
            if isinstance(p, dict)
            and "query" in p and "pos" in p
            and len(str(p["query"])) > 15
            and len(str(p["pos"]))   > 30
        ]
        if valid:
            return valid
        time.sleep(1.0)
    return []

def restart_ollama_if_needed():
    import subprocess, requests
    try:
        requests.get("http://localhost:11434", timeout=3)
        return True
    except:
        print("\n⚠️  Ollama crashed — restarting...")
        subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
        time.sleep(3)
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            env=os.environ.copy()
        )
        for _ in range(15):
            try:
                requests.get("http://localhost:11434", timeout=3)
                print("✅ Ollama restarted!")
                return True
            except:
                time.sleep(3)
        return False

failed    = 0
save_file = open(PAIRS_FILE, "a", encoding="utf-8")

print(f"\n🔄 Generating Q&A pairs...")
print(f"   Total   : {len(chunks)} chunks")
print(f"   Done    : {len(done_indices)}")
print(f"   Left    : {len(chunks) - len(done_indices)}\n")

for i, chunk in enumerate(chunks):
    if i in done_indices:
        continue

    if i % 50 == 0:
        restart_ollama_if_needed()

    result = generate_pairs_for_chunk(chunk, n=QUERIES_PER_CHUNK)
    if result:
        for query, c in result:
            pairs.append((query, c))
            save_file.write(json.dumps({
                "chunk_idx": i,
                "query"    : query,
                "chunk"    : c
            }, ensure_ascii=False) + "\n")
        save_file.flush()
        done_indices.add(i)
    else:
        failed += 1

    if (i+1) % 10 == 0:
        pct = (i+1)/len(chunks)*100
        print(f"   [{i+1:3d}/{len(chunks)}] {pct:.0f}%"
              f" | pairs: {len(pairs)} | failed: {failed}")

save_file.close()
random.shuffle(pairs)
print(f"\n✅ Total pairs : {len(pairs)}")
print(f"   Failed      : {failed}")
print(f"   Done chunks : {len(done_indices)}/{len(chunks)}")

In [ ]:
import numpy as np
from FlagEmbedding import FlagModel
from rank_bm25 import BM25Okapi

print("🔄 Building BM25 index...")
tokenized_corpus = [c.lower().split() for c in chunks]
bm25             = BM25Okapi(tokenized_corpus)
print("✅ BM25 ready!")

print("\n🔄 Loading BGE for dense mining...")
miner       = FlagModel("BAAI/bge-large-en-v1.5",
                         query_instruction_for_retrieval=QUERY_INSTRUCTION,
                         use_fp16=True)

print("🔄 Encoding corpus...")
corpus_embs = miner.encode(chunks, batch_size=32)
corpus_embs = corpus_embs / np.linalg.norm(corpus_embs, axis=1, keepdims=True)
print(f"✅ Dense index: {corpus_embs.shape}")

def get_hybrid_negatives(query, pos_text):
    try:
        true_idx = chunks.index(pos_text)
    except ValueError:
        return None

    bm25_scores            = bm25.get_scores(query.lower().split())
    bm25_scores[true_idx]  = -1
    bm25_top               = np.argsort(bm25_scores)[::-1]
    bm25_negs              = [chunks[i] for i in bm25_top
                               if i != true_idx and bm25_scores[i] > 0][:BM25_NEGS]

    q_emb                  = miner.encode_queries([query])
    q_emb                  = q_emb / np.linalg.norm(q_emb)
    dense_scores           = (q_emb @ corpus_embs.T)[0]
    dense_scores[true_idx] = -1
    dense_top              = np.argsort(dense_scores)[::-1]
    dense_negs             = [chunks[i] for i in dense_top
                               if i != true_idx][:DENSE_NEGS + 5]

    seen, negs = set([pos_text]), []
    for neg in bm25_negs + dense_negs:
        if neg not in seen:
            seen.add(neg)
            negs.append(neg)
        if len(negs) >= HARD_NEG_PER_QUERY:
            break

    return negs if len(negs) >= 2 else None

print("\n🔄 Mining hard negatives...")
triplets = []
skipped  = 0

for i, (query, pos_text) in enumerate(pairs):
    negs = get_hybrid_negatives(query, pos_text)
    if negs:
        triplets.append({
            "query": query,
            "pos"  : [pos_text],
            "neg"  : negs
        })
    else:
        skipped += 1
    if (i+1) % 100 == 0:
        print(f"   [{i+1}/{len(pairs)}] triplets: {len(triplets)} | skipped: {skipped}")

del miner
import torch, gc
gc.collect()
torch.cuda.empty_cache()

print(f"\n✅ Triplets : {len(triplets)}")
print(f"   Skipped  : {skipped}")

In [ ]:
import json, random, os

random.shuffle(triplets)
split      = int(len(triplets) * 0.9)
train_data = triplets[:split]
eval_data  = triplets[split:]

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"   ✅ {os.path.basename(path)} — {len(data)} records ({os.path.getsize(path)/1024:.0f} KB)")

print("💾 Saving...")
save_jsonl(train_data, "/content/train_data.jsonl")
save_jsonl(eval_data,  "/content/eval_data.jsonl")

with open("/content/chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps({"pos": [c]}, ensure_ascii=False) + "\n")
print(f"   ✅ chunks.jsonl — {len(chunks)} chunks")
print(f"\n   Train : {len(train_data)}")
print(f"   Eval  : {len(eval_data)}")

In [ ]:
import subprocess, torch, gc, time

print("🔄 Stopping Ollama...")
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(3)
gc.collect()
torch.cuda.empty_cache()

free = (torch.cuda.get_device_properties(0).total_memory
        - torch.cuda.memory_allocated()) / 1e9
print(f"✅ VRAM free: {free:.1f} GB")
print("✅ Ready for fine-tuning!")

In [ ]:
!python -m FlagEmbedding.finetune.embedder.encoder_only.base \
    --model_name_or_path              BAAI/bge-large-en-v1.5 \
    --train_data                      /content/train_data.jsonl \
    --output_dir                      /content/finetuned-bge-nokia \
    --learning_rate                   8e-6 \
    --fp16 \
    --num_train_epochs                10 \
    --per_device_train_batch_size     4 \
    --gradient_accumulation_steps     8 \
    --gradient_checkpointing \
    --dataloader_drop_last            True \
    --query_max_len                   96 \
    --passage_max_len                 512 \
    --temperature                     0.02 \
    --query_instruction_for_retrieval "Represent this sentence for searching relevant passages:" \
    --warmup_ratio                    0.1 \
    --weight_decay                    0.01 \
    --logging_steps                   5 \
    --save_steps                      100 \
    --report_to                       none

In [ ]:
import numpy as np, json, torch
from FlagEmbedding import FlagModel

eval_samples = []
with open("/content/eval_data.jsonl") as f:
    for line in f:
        eval_samples.append(json.loads(line.strip()))
print(f"✅ Eval samples: {len(eval_samples)}")

def evaluate_model(model_path, data, label):
    print(f"\n{'='*50}\nEvaluating: {label}\n{'='*50}")
    model    = FlagModel(model_path,
                         query_instruction_for_retrieval=QUERY_INSTRUCTION,
                         use_fp16=True)
    passages = list(dict.fromkeys([t["pos"][0] for t in data]))
    p_embs   = model.encode(passages, batch_size=16)
    p_embs   = p_embs / np.linalg.norm(p_embs, axis=1, keepdims=True)

    mrr, r1, r5, r10 = [], [], [], []
    for t in data:
        try:
            ti = passages.index(t["pos"][0])
        except ValueError:
            continue
        q_emb  = model.encode_queries([t["query"]])
        q_emb  = q_emb / np.linalg.norm(q_emb)
        scores = (q_emb @ p_embs.T)[0]
        ranked = list(np.argsort(scores)[::-1][:10])
        try:
            mrr.append(1.0 / (ranked.index(ti) + 1))
        except ValueError:
            mrr.append(0.0)
        r1.append (1.0 if ti in ranked[:1]  else 0.0)
        r5.append (1.0 if ti in ranked[:5]  else 0.0)
        r10.append(1.0 if ti in ranked[:10] else 0.0)

    res = {"MRR@10": float(np.mean(mrr)), "R@1": float(np.mean(r1)),
           "R@5"   : float(np.mean(r5)),  "R@10": float(np.mean(r10))}
    for k, v in res.items():
        print(f"  {k:10}: {v:.4f}")
    del model
    torch.cuda.empty_cache()
    return res

base = evaluate_model("BAAI/bge-large-en-v1.5",       eval_samples, "BASE MODEL")
ft   = evaluate_model("/content/finetuned-bge-nokia",  eval_samples, "FINE-TUNED")

print(f"\n{'='*55}\nIMPROVEMENT SUMMARY\n{'='*55}")
for m in ["MRR@10", "R@1", "R@5", "R@10"]:
    d = ft[m] - base[m]
    print(f"{m:10}  Base:{base[m]:.4f}  FT:{ft[m]:.4f}  {'↑' if d>0 else '↓'} {d:+.4f}")

In [ ]:
import os, shutil, subprocess
from google.colab import files

# Free disk space
paths_to_delete = [
    "/root/.ollama/models",
    "/tmp/ollama.tgz",
    "/root/.cache/huggingface/hub",
]
for step in range(100, 1100, 100):
    paths_to_delete.append(f"/content/finetuned-bge-nokia/checkpoint-{step}")

for path in paths_to_delete:
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
        print(f"🗑️  Deleted: {path}")

result = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
print(f"\n📊 Disk:\n{result.stdout}")

# Zip & download
print("🔄 Zipping model...")
shutil.make_archive("/content/finetuned-bge-nokia", "zip",
                    "/content/finetuned-bge-nokia")

for fp in [
    "/content/finetuned-bge-nokia.zip",
    "/content/train_data.jsonl",
    "/content/eval_data.jsonl",
    "/content/chunks.jsonl",
    "/content/pairs_progress.jsonl",
    "/content/chunks_backup.jsonl",
]:
    if os.path.exists(fp):
        size = os.path.getsize(fp)/1e6
        print(f"⬇️  {os.path.basename(fp)} ({size:.1f} MB)")
        files.download(fp)

print("\n🎉 ALL DONE!")


## Run Order
```
Cell 1  → Restart Session ⚠️
Cell 2  → verify packages
Cell 3  → install Ollama
Cell 4  → pull llama3.1:8b
Cell 5  → config (set PDF path)
Cell 6  → chunk PDF (upload PDF first)
Cell 7  → generate pairs (~2hrs, auto-saves)
Cell 8  → hard negatives
Cell 9  → save data
Cell 10 → free GPU
Cell 11 → fine-tune (~45 mins)
Cell 12 → evaluate
Cell 13 → download everything